# P76 — Un estudio de la validación cruzada y el bootstrap para estimar exactitud y seleccionar modelos

## 1. Título y paper

**Paper:** *A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection*  
**Autoría:** Ron Kohavi  
**Año y venue:** 1995 · IJCAI'95, 1137–1143  
**Nivel:** L3 · **Motor:** `validacion_cruzada`  
**Ficha completa:** [`P76_validacion_cruzada`](../../papers/foundational/P76_validacion_cruzada/README.md)

**Hito:** Fija la práctica estándar de evaluación —diez pliegues estratificados— con evidencia empírica en lugar de costumbre.

- [Actas IJCAI'95 (PDF)](https://www.ijcai.org/Proceedings/95-2/Papers/016.pdf)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Se reportaban exactitudes sin decir cómo se habían estimado. Holdout, validación cruzada y bootstrap dan números distintos sobre los mismos datos, y nadie había medido cuál era preferible ni por qué.
2. Ejecutar una implementación mínima de la propuesta: Comparar empíricamente los estimadores en sesgo y varianza sobre conjuntos reales, y recomendar validación cruzada estratificada de diez pliegues como compromiso entre ambos.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Stone (1974), validación cruzada
- Efron (1979), bootstrap


## 4. Intuición

Dos personas evalúan el mismo modelo sobre los mismos datos y reportan 0,70 y 0,97. Ninguna hace trampa. La diferencia está en cómo partieron los datos, y ese detalle —que casi nunca se declara— pesa más que muchas mejoras publicadas.


## 5. Concepto mínimo

```text
Holdout 70/30       : entrena con 70, evalúa con 30      ← 30 ejemplos de test
Validación cruzada k : k particiones; cada ejemplo pasa por
                       el test EXACTAMENTE una vez             ← n ejemplos de test

Mismo sesgo aproximado.  La diferencia está en la VARIANZA del estimador.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('validacion_cruzada', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Acertarán los tres estimadores la exactitud real en media?
2. ¿Cuál tendrá más dispersión?
3. ¿Por qué?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('validacion_cruzada', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('validacion_cruzada', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Los tres aciertan en media —los sesgos son de milésimas—, así que el problema no es el sesgo. Es la **varianza**: el holdout estima con desviación 0,0753 y la validación cruzada de 10 pliegues con 0,0454. Sobre 200 conjuntos simulados, el holdout devuelve valores entre **0,60 y 0,97**.


## 10. Comentario pedagógico

La razón es aritmética y conviene tenerla clara: el holdout evalúa sobre 30 ejemplos y la validación cruzada sobre los 100, porque cada ejemplo pasa por el test una vez. Menos ejemplos de test, más ruido. Por eso el artículo recomienda diez pliegues estratificados — no por tradición, por medición.


## 11. Error o anti-patrón deliberado

Anti-patrón: reportar una exactitud sin decir cómo se estimó.


In [ ]:
print('«El modelo alcanza un 92% de exactitud».')
print('Sin decir: particion, semilla, numero de corridas ni estratificacion.')
print('Con holdout sobre 30 ejemplos de test, ese 92% puede ser un 78% con otra particion.')

## 12. Corrección

El reporte mínimo que hace comparable un número:


In [ ]:
r = run_paper_lab('validacion_cruzada', seed=7)['result']
for nombre, e in r['estimadores'].items():
    print(f"{nombre:<22} media={e['media']:<8} desv={e['desviacion']:<8} "
          f"rango=[{e['min']}, {e['max']}]")
print('exactitud real de la poblacion:', r['exactitud_real_de_la_poblacion'])

## 13. Desafío guiado

Calcula cuántas veces más disperso es el holdout que la validación cruzada de 10 pliegues, y relaciónalo con el número de ejemplos de test de cada uno.


In [ ]:
r = run_paper_lab('validacion_cruzada', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma un modelo tuyo y evalúalo con holdout repetido 20 veces y con validación cruzada de 10 pliegues. Publica media, desviación y número de corridas de ambos, y decide cuál reportarías.


## 15. Evidencia de aprendizaje

Guarda la tabla de los tres estimadores con media, desviación y rango, y tu regla de reporte mínimo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P76_validacion_cruzada/README.md) · evaluación formal: [`assessments/papers/P76_validacion_cruzada.md`](../../assessments/papers/P76_validacion_cruzada.md)


## 16. Cierre

Ya se sabe medir. Volvemos al modelo: cómo evitar que use todas las variables que le des, incluso las que no aportan.


## 17. Conexión con el siguiente hito

- P63
- P80

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
